##  Experiments

In [5]:
import json
import numpy as np
import pandas as pd
import os
from google.cloud import bigquery
from google.cloud import storage
import joblib
from io import BytesIO
from dotenv import load_dotenv
import types
import sys

pd.set_option("display.max_columns", None)

In [2]:
load_dotenv()

project_id = os.getenv("GCP_PROJECT_ID")
bucket_name = os.getenv("GCP_BUCKET_NAME")
model_ruta = os.getenv("MODEL_ROOT")
pipeline_ruta = os.getenv("PIPELINE_ROOT")
credentials_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Tablas inputs
project_id_input = os.getenv("GCP_PROJECT_ID_INPUT")
dataset_id_input = os.getenv("BQ_DATASET_ID_INPUT")
table_id_input = os.getenv("BQ_TABLE_ID_INPUT")

# Tablas features
dataset_id_features = os.getenv("BQ_DATASET_ID_FEATURES")
table_id_features = os.getenv("BQ_TABLE_ID_FEATURES")

In [3]:
model_name  = "Model-GradientBoostingRegressor.joblib"
model_path = f"{model_ruta}/{model_name}"
print(model_path)

pipeline_name = 'Pipeline-Transformacion-Training.joblib'
pipeline_path = f"{pipeline_ruta}/{pipeline_name}"
print(pipeline_path)

metadata_name = 'metadata_transformer.py'
metadata_path = f"{pipeline_ruta}/{metadata_name}"
print(metadata_path)

project-pipeline-predictions-casas/models/Model-GradientBoostingRegressor.joblib
project-pipeline-predictions-casas/pipeline/Pipeline-Transformacion-Training.joblib
project-pipeline-predictions-casas/pipeline/metadata_transformer.py


In [4]:
# Cliente de Cloud Storage
client = storage.Client(project=project_id)

bucket = client.bucket(bucket_name)
blob = bucket.blob(model_path)

contenido_joblib  = blob.download_as_bytes()

modelo = joblib.load(BytesIO(contenido_joblib))

print("Archivo JOBLIB cargado correctamente ✅")
print(type(modelo))

Archivo JOBLIB cargado correctamente ✅
<class 'sklearn.ensemble._gb.GradientBoostingRegressor'>


D:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator DummyRegressor from version 1.9.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
D:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.9.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
D:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages

In [6]:
blob = bucket.blob(metadata_path)

module_code  = blob.download_as_text(encoding="utf-8")


metadata_transformer = types.ModuleType('metadata_transformer')
metadata_transformer.__file__ = "gs://.../metadata_transformer.py"
sys.modules['metadata_transformer'] = metadata_transformer

exec(compile(module_code, metadata_transformer.__file__, "exec"),
     metadata_transformer.__dict__)

In [7]:
blob = bucket.blob(pipeline_path)

contenido_joblib  = blob.download_as_bytes()

pipeline = joblib.load(BytesIO(contenido_joblib))

print("Archivo JOBLIB cargado correctamente ✅")
print(type(pipeline))

Archivo JOBLIB cargado correctamente ✅
<class 'sklearn.pipeline.Pipeline'>


D:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.9.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
D:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.9.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [8]:
client = bigquery.Client(project=project_id)

path_bq_table = f"{project_id_input}.{dataset_id_input}.{table_id_input}"

dfInput = client.query(
        f'''SELECT * FROM `{path_bq_table}`
        '''
    ).to_dataframe()

path_bq_table_feature = f"{project_id}.{dataset_id_features}.{table_id_features}"
dfFeatures = client.query(
        f'''SELECT * FROM `{path_bq_table_feature}`
        '''
    ).to_dataframe()

listFeatures = dfFeatures['features'].tolist()

dfInputFeatures = dfInput[listFeatures].copy()
dfInputFeatures


D:\03_PROYECTOS\Project-GCP-Vertex-AI-Pipelines\project-data-processing\.venv\lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,mssubclass,mszoning,lotfrontage,lotarea,neighborhood,overallqual,overallcond,yearbuilt,yearremodadd,bsmtqual,bsmtfintype_principal,bsmtfinsf_principal,totalbsmtsf,centralair,firstflrsf,secondflrsf,grlivarea,kitchenqual,fireplacequ,garagetype,garagecars,garagearea,yrsold
0,20,RH,80.0,11622,NAmes,5,6,1961,1961,TA,Rec,468.0,882.0,Y,896.0,0.0,896.0,TA,None,Attchd,1.0,730.0,2010
1,20,RL,81.0,14267,NAmes,6,6,1958,1958,TA,ALQ,923.0,1329.0,Y,1329.0,0.0,1329.0,Gd,None,Attchd,1.0,312.0,2010
2,60,RL,74.0,13830,Gilbert,5,5,1997,1998,Gd,GLQ,791.0,928.0,Y,928.0,701.0,1629.0,TA,TA,Attchd,2.0,482.0,2010
3,60,RL,78.0,9978,Gilbert,6,6,1998,1998,TA,GLQ,602.0,926.0,Y,926.0,678.0,1604.0,Gd,Gd,Attchd,2.0,470.0,2010
4,120,RL,43.0,5005,StoneBr,8,5,1992,1992,Gd,ALQ,263.0,1280.0,Y,1280.0,0.0,1280.0,Gd,None,Attchd,2.0,506.0,2010
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,160,RM,21.0,1936,MeadowV,4,7,1970,1970,TA,Unf,0.0,546.0,Y,546.0,546.0,1092.0,TA,None,None,0.0,0.0,2006
1455,160,RM,21.0,1894,MeadowV,4,5,1970,1970,TA,Rec,252.0,546.0,Y,546.0,546.0,1092.0,TA,None,CarPort,1.0,286.0,2006
1456,20,RL,160.0,20000,Mitchel,5,7,1960,1996,TA,ALQ,1224.0,1224.0,Y,1224.0,0.0,1224.0,TA,TA,Detchd,2.0,576.0,2006
1457,85,RL,62.0,10441,Mitchel,5,5,1992,1992,Gd,GLQ,337.0,912.0,Y,970.0,0.0,970.0,TA,None,None,0.0,0.0,2006
